# 为什么拓展

实战项目1中，表格完全被指定，prompt也写死，相当于只是固定表格的分析Agent。于是做了一个小改动，让用户可以上传任意表格

# 表格处理

## 导入依赖和模型

In [1]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-chat",
    base_url="https://api.deepseek.com",
    api_key=""
)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 导入表格

In [2]:
import os
import pandas as pd


def load_dataframe(file_path: str):

    suffix = os.path.splitext(file_path)[1].lower()

    if suffix == ".csv":
        df = pd.read_csv(file_path)

    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            f"不支持的文件类型：{suffix}"
        )

    return df

## 表格数据读取

In [3]:
def build_dataset_profile(df: pd.DataFrame):

    try:
        sample = df.head(5).to_markdown()
    except Exception:
        sample = df.head(5).to_string()

    profile = f"""
数据集信息：

行数：
{len(df)}

列数：
{len(df.columns)}

字段列表：
{list(df.columns)}

字段类型：
{df.dtypes.to_string()}

前5行样例数据：

{sample}
"""

    return profile

## 上传表格

In [6]:
file_path = "SIMT User Story.xlsx"

df = load_dataframe(file_path)

print(
    f"加载成功，共 {len(df)} 行，{len(df.columns)} 列"
)

加载成功，共 13 行，7 列


In [7]:
df.head()

,Issue Type,Key,Summary,Assignee,Priority,Status,Resolution
0,Task,CT-14110,Automation task_Sameera Sprint1,Sameera De Silva,Medium,Ready for Planning,NaN
1,Task,CT-14074,Automation task_Winky,Winky Nan,Medium,Ready for Planning,NaN
2,Task,CT-14069,Automation task_Lily_Sprint1,Lily Tian,Medium,Ready for Planning,NaN
3,Task,CT-14042,FY27Q1 Sprint1 Prod Support,Priyanga Arachchi,P3 (Normal),Ready for Planning,NaN
4,Task,CT-14015,Optimize of Dates of filter panel on overview ...,Elwin Sun,High,Ready for Planning,NaN


In [8]:
profile = build_dataset_profile(df)

print(profile)


数据集信息：

行数：
13

列数：
7

字段列表：
['Issue Type', 'Key', 'Summary', 'Assignee', 'Priority', 'Status', 'Resolution']

字段类型：
Issue Type     object
Key            object
Summary        object
Assignee       object
Priority       object
Status         object
Resolution    float64

前5行样例数据：

  Issue Type       Key                                                                Summary           Assignee     Priority              Status  Resolution
0       Task  CT-14110                                        Automation task_Sameera Sprint1   Sameera De Silva       Medium  Ready for Planning         NaN
1       Task  CT-14074                                                  Automation task_Winky          Winky Nan       Medium  Ready for Planning         NaN
2       Task  CT-14069                                           Automation task_Lily_Sprint1          Lily Tian       Medium  Ready for Planning         NaN
3       Task  CT-14042                                            FY27Q1 Sprint1 Prod

# Agent构造

## Tool

In [9]:
from langchain_experimental.tools import PythonAstREPLTool

tool = PythonAstREPLTool(
    locals={
        "df": df,
        "pd": pd
    }
)

C:\Users\yanchen\AppData\Local\Temp\ipykernel_30532\3423552611.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonAstREPLTool


## prompt

In [10]:
system_prompt = f"""
你是一名专业的数据分析师。

当前已经加载了一个DataFrame。

变量名：

df

请始终使用df进行分析。

以下是数据集描述：

{profile}

工作规则：

1. 所有结论必须来自真实数据
2. 如果需要统计请编写Python代码
3. 如果需要计算请编写Python代码
4. 如果需要绘图请使用matplotlib
5. 不允许臆测数据
6. 优先调用Python工具分析后再回答
"""

## 定义返回格式

In [11]:
from pydantic import BaseModel,Field

class AnalysisResult(BaseModel):
    question: str = Field(description="用户问题")

    code_used: str = Field(
        description="Agent实际使用的分析代码"
    )

    result: str = Field(
        description="分析结果"
    )

    insight: str = Field(
        description="分析洞察"
    )

## Create agent

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[tool],
    system_prompt=system_prompt,
    response_format=AnalysisResult
)

# 测试

## 结果

In [13]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "sheet1中type为story有多少个？"
            }
        ]
    }
)

In [15]:
result = response["structured_response"]

print(result.question)
print(result.code_used)
print(result.result)
print(result.insight)

sheet1中type为story有多少个？
import pandas as pd
story_count = df[df['Issue Type'] == 'Story'].shape[0]
story_count
5
在数据集中，Issue Type为'Story'的记录共有5条。
